In [1]:
import os
os.chdir('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d')
print(os.listdir('.'))

['ldh_fast_eval.py', 'ldh_def.py', '.config', '.prompts', 'memory', '.kernel_llm_logs_1.txt']


In [2]:
import sys, time
sys.path.insert(0, '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d')
import mpmath
mpmath.mp.dps = 50

# Test import
from ldh_fast_eval import Z_DH_fast, init_evaluator, set_height, L_DH_fast

# Quick benchmark
t0 = time.time()
val = Z_DH_fast(100.1)
print(f"Z_DH_fast(100.1) = {val}, took {time.time()-t0:.3f}s")
t0 = time.time()
val = Z_DH_fast(500.2)
print(f"Z_DH_fast(500.2) = {val}, took {time.time()-t0:.3f}s")


ModuleNotFoundError: No module named 'cypari2'

In [3]:
import subprocess
res = subprocess.run(['pip', 'install', 'cypari2'], capture_output=True, text=True)
print(res.stdout[-2000:])
print("STDERR:", res.stderr[-2000:])


STDERR: 


In [4]:
import sys, time
sys.path.insert(0, '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d')
import mpmath
mpmath.mp.dps = 50

from ldh_fast_eval import Z_DH_fast, init_evaluator, set_height, L_DH_fast

t0 = time.time()
val = Z_DH_fast(100.1)
print(f"Z_DH_fast(100.1) = {val}, took {time.time()-t0:.3f}s")
t0 = time.time()
val = Z_DH_fast(500.2)
print(f"Z_DH_fast(500.2) = {val}, took {time.time()-t0:.3f}s")
t0 = time.time()
val = Z_DH_fast(1000.3)
print(f"Z_DH_fast(1000.3) = {val}, took {time.time()-t0:.3f}s")

PARI stack size set to 536870912 bytes, maximum size set to 536870912


Z_DH_fast(100.1) = 0.9692404914060999106801890495172138454118125759798, took 0.006s
Z_DH_fast(500.2) = -1.4285812115286039808137032366234994000361649927214, took 0.008s
Z_DH_fast(1000.3) = 1.0620466873042933991263773845399304724947185607768, took 0.014s


In [5]:
# Understand the expected zero density to estimate work.
# For L_DH: N(T) ~ (T/pi) log(T/(2 pi e/sqrt(5))) - approximately
# Let's empirically estimate by scanning and look at where ~5000th zero is.
# Spacing ~ 2pi / log(T/(2pi)) at large T. For zeta, zero #5000 is near T~3293.
# For L_DH with conductor 5, density is slightly higher: density at T is ~ (1/pi) log(5T/(2pi))
# Solve: N(T)=5000. Let's estimate.

import mpmath
from mpmath import mpf, pi, log

# Approximate N(T) for L_DH (degree 1 over chi mod 5, but as combination has functional equation like zeta with conductor 5)
# N(T) ~ T/(2pi) * log(5 T / (2 pi e))
def N_T(T):
 T = mpf(T)
 return T/(2*pi) * log(5*T/(2*pi*mpmath.e))

for T in [1000, 2000, 2500, 3000, 3500]:
 print(T, float(N_T(T)))

1000 903.8912346763555
2000 2028.4180695053624
2500 2624.308584908544
3000 3236.222432878156
3500 3861.4612877220434


In [6]:
# So 5000 zeros for L_DH should be reached around T~4300 or so.
# Let's calculate more precisely:
for T in [4000, 4200, 4400, 4500]:
 print(T, float(N_T(T)))

4000 4498.107339316028
4200 4755.626528649516
4400 5014.662052716108
4500 5144.726642247402


In [7]:
# Plan: Initialize lfuninit at T_max=5000 (or higher for safety), scan with small step size to find sign changes, refine each.
# Step size: spacing at T~4400 is 2*pi/log(5*T/(2pi)) ~ 2pi/log(3501) ~ 6.28/8.16 ~ 0.77.
# Use step=0.05 or so to avoid missing closely spaced or off-line pairs.

# But L_DH has off-line zeros (Davenport-Heilbronn does NOT satisfy RH). Spira found zero near 0.808 + 85.7i.
# The objective says "first 5,000 critical-line zeros". We need critical-line zeros (sign changes of Z_DH on the line).
# Since L_DH has off-line zeros, the number of zeros on the line is fewer than total nontrivial zeros.
# But our N(T) above counts all nontrivial zeros. We need to find 5000 critical-line zeros (=sign changes of Z_DH).
# We'll iterate sign-change finding until count = 5000.

# Let's first set up: re-init at higher T_max
import importlib, ldh_fast_eval
importlib.reload(ldh_fast_eval)
from ldh_fast_eval import Z_DH_fast, init_evaluator, set_height, L_DH_fast
print("re-init at T_max=5200")
init_evaluator(T_max=5200, dps=80)
print("done")

PARI stack size set to 536870912 bytes, maximum size set to 536870912


re-init at T_max=5200


done


In [8]:
import time
t0=time.time()
v = Z_DH_fast(4500.123)
print(v, "took", time.time()-t0)

0.93577378605450128288936448581842905005141046330141 took 0.14257240295410156


In [9]:
# 0.14s at T~4500 is significant for many evaluations. We need to be efficient.
# Estimate budget: at avg ~50ms per eval, with step 0.05 over T=0..4400 -> 88000 scans. Plus refinements ~ 50 evals each * 5000 = 250000 refinements.
# Total ~ 338,000 evals * 50ms = 4.7 hours. Too slow.
# 
# Better approach: use mpmath at lower precision for scanning (use working dps lower in PARI). But the inner _pari is at dps=80.
# We can do scan at lower precision and refine at high precision.

# Actually let's check time scaling vs T:
import time
for T in [100, 500, 1000, 2000, 3000, 4000, 4500]:
 t0=time.time()
 v = Z_DH_fast(T + 0.123)
 dt = time.time()-t0
 print(f"T={T}: {dt*1000:.1f} ms")

T=100: 7.6 ms
T=500: 7.7 ms
T=1000: 14.0 ms
T=2000: 54.4 ms


T=3000: 143.4 ms
T=4000: 139.0 ms
T=4500: 138.9 ms


In [10]:
# So time grows but caps around 140ms at high T. Let's see if we can reduce dps for scanning.
# The PARI working precision is dps=80. For scanning sign changes we only need sign, not high precision.
# Let's try a lower dps PARI init.

# Reinit at dps=30 PARI for scanning
init_evaluator(T_max=5200, dps=30)
import time
for T in [100, 500, 1000, 2000, 3000, 4000, 4500]:
 t0=time.time()
 v = Z_DH_fast(T + 0.123)
 dt = time.time()-t0
 print(f"T={T}: {dt*1000:.1f} ms, val={float(v):.4f}")

T=100: 6.3 ms, val=0.9780
T=500: 6.7 ms, val=-1.6749
T=1000: 16.0 ms, val=1.1875
T=2000: 41.1 ms, val=-0.8095
T=3000: 40.4 ms, val=-1.0625
T=4000: 40.3 ms, val=2.2665
T=4500: 39.3 ms, val=0.9358


In [11]:
# Better. Now: scanning ~88,000 evals at avg 25ms ~ 37min. Plus refinement: 5000 zeros * ~20 evals = 100K evals.
# We need to refine at high precision (dps=50) per objective. So switch precision dynamically.
# Strategy:
# 1) Scan with dps=30 (PARI), step h~0.05
# 2) For each bracket, refine with mpmath.findroot using a high-precision evaluator (re-init PARI to dps=80 only for refinement).
# Re-initing PARI is fast actually - it just sets default.

# But re-initing _pari("default(realprecision, 30)") on existing lfuninit caches: does lfuninit's stored cache encode precision? Yes, lfuninit precomputes at the precision that was current. So changing it after may not help fully.

# Let's check: if we built lfuninit at dps=30 then call lfun at dps=80, does precision degrade? Probably yes - cached coefficients have only 30 digits.
# Strategy: scan at dps=30 init with cheap setup. Then for refinement, switch to a high-dps init.

# Actually we can have two PARI scopes. But the module has one global. Let me think...
# Simpler: scan + refinement both at dps=80 = ~140ms per eval. 88000 scans = 12300s ~ 3.4 hours just scanning. Too slow.

# Better strategy: 
# 1) Run with dps=30 init for scanning - get sign of Z_DH.
# 2) For each detected zero, set_height + reinit at dps=80 for the refinement (re-do lfuninit at dps=80 once - expensive).
# Actually, simplest is: do entire work at PARI dps=50 (which is what target precision is). 50 digits should still be much faster than 80.

# Let's test:
init_evaluator(T_max=5200, dps=55)
import time
for T in [100, 500, 1000, 2000, 3000, 4000, 4500]:
 t0=time.time()
 v = Z_DH_fast(T + 0.123)
 dt = time.time()-t0
 print(f"T={T}: {dt*1000:.1f} ms, val={mpmath.nstr(v,8)}")

T=100: 5.4 ms, val=0.97798579
T=500: 6.5 ms, val=-1.6748653
T=1000: 11.3 ms, val=1.1874581
T=2000: 44.0 ms, val=-0.80952468
T=3000: 63.5 ms, val=-1.0624822
T=4000: 62.6 ms, val=2.2664746


T=4500: 65.4 ms, val=0.93577379


In [12]:
# 65ms at high T. dps=55 too high. Let's split: do scan at dps=30 init, refine at dps=80 init.
# But re-init is expensive (we'd reinit per zero ~5000 times). Bad.

# Best approach: TWO separate cypari instances? cypari2.Pari() is singleton but maybe we can use ONE Pari instance with multiple lfuninit objects at different precision.
# Actually default(realprecision, X) affects subsequent lfuninit. We can do lfuninit at dps=30 AND dps=80 and store both.

# Modify approach: maintain Lchi_init_lo (low precision) and Lchi_init_hi (high precision).

# Let me write a custom version inline.
import cypari2, mpmath, time
pari = cypari2.Pari()
pari.allocatemem(512*1024*1024)

# Initialize both
pari("default(realprecision, 80)")
pari("gz5 = znstar(5, 1)")
pari("Lchi = lfuncreate([gz5, [1]])")
pari("Lchibar = lfuncreate([gz5, [3]])")

print("Building lfuninit at dps=30 ...")
pari("default(realprecision, 30)")
pari("Lchi_lo = lfuninit(Lchi, [5200])")
pari("Lchibar_lo = lfuninit(Lchibar, [5200])")

print("Building lfuninit at dps=80 ...")
pari("default(realprecision, 80)")
pari("Lchi_hi = lfuninit(Lchi, [5200])")
pari("Lchibar_hi = lfuninit(Lchibar, [5200])")
print("done")

PARI stack size set to 536870912 bytes, maximum size set to 536870912
Building lfuninit at dps=30 ...


Building lfuninit at dps=80 ...
done


In [13]:
mpmath.mp.dps = 50
sqrt5 = mpmath.sqrt(5)
xi_dh = (mpmath.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
I = mpmath.mpc(0,1)
A_const = (1 - I*xi_dh)/2
B_const = (1 + I*xi_dh)/2

def _real(name):
 return mpmath.mpf(str(pari("real(%s)"%name)))
def _imag(name):
 return mpmath.mpf(str(pari("imag(%s)"%name)))

def theta_dh(t):
 t = mpmath.mpf(t)
 return (t/2)*mpmath.log(5/mpmath.pi) + mpmath.im(mpmath.loggamma(mpmath.mpc(mpmath.mpf("0.75"), t/2)))

def Z_lo(t):
 """Low-precision Z_DH evaluation using dps=30 lfuninit."""
 t = mpmath.mpf(t)
 s_str = f"(0.5) + ({mpmath.nstr(t, 30, strip_zeros=False)})*I"
 pari(f"_v1 = lfun(Lchi_lo, {s_str})")
 pari(f"_v2 = lfun(Lchibar_lo, {s_str})")
 Lc = mpmath.mpc(_real("_v1"), _imag("_v1"))
 Lcb = mpmath.mpc(_real("_v2"), _imag("_v2"))
 Ldh = A_const*Lc + B_const*Lcb
 return mpmath.re(mpmath.exp(I*theta_dh(t)) * Ldh)

def Z_hi(t):
 """High-precision Z_DH evaluation using dps=80 lfuninit."""
 t = mpmath.mpf(t)
 s_str = f"(0.5) + ({mpmath.nstr(t, 70, strip_zeros=False)})*I"
 pari(f"_v1 = lfun(Lchi_hi, {s_str})")
 pari(f"_v2 = lfun(Lchibar_hi, {s_str})")
 Lc = mpmath.mpc(_real("_v1"), _imag("_v1"))
 Lcb = mpmath.mpc(_real("_v2"), _imag("_v2"))
 Ldh = A_const*Lc + B_const*Lcb
 return mpmath.re(mpmath.exp(I*theta_dh(t)) * Ldh)

# benchmark both
for T in [100, 1000, 2000, 3000, 4000, 4500]:
 t0=time.time()
 v = Z_lo(T+0.123)
 dt = time.time()-t0
 t1=time.time()
 v2 = Z_hi(T+0.123)
 dt2 = time.time()-t1
 print(f"T={T}: Z_lo={dt*1000:.1f}ms, Z_hi={dt2*1000:.1f}ms, vals: {mpmath.nstr(v,6)} vs {mpmath.nstr(v2,12)}")

PariError: syntax error, unexpected invalid token, expecting end of file

In [14]:
# nstr at dps=30 may use scientific notation with "e". Let me test what it returns:
print(repr(mpmath.nstr(mpmath.mpf("100.123"), 30, strip_zeros=False)))
print(repr(mpmath.nstr(mpmath.mpf("4500.123"), 30, strip_zeros=False)))
# Could be the issue is the s_str format - let's look at the original library
print(mpmath.nstr(mpmath.mpf("1e-20"), 30, strip_zeros=False))

'100.123000000000000000000000000'
'4500.12300000000000000000000000'
1.00000000000000000000000000000e-20


In [15]:
# Hmm those look fine. Maybe the issue is that I used `_v1` as variable name and underscore prefix is invalid in PARI?
# In PARI/GP, _ is not a valid identifier start. Let me rename.
def Z_lo(t):
 t = mpmath.mpf(t)
 s_str = f"(0.5) + ({mpmath.nstr(t, 30, strip_zeros=False)})*I"
 pari(f"v1 = lfun(Lchi_lo, {s_str})")
 pari(f"v2 = lfun(Lchibar_lo, {s_str})")
 Lc = mpmath.mpc(mpmath.mpf(str(pari("real(v1)"))), mpmath.mpf(str(pari("imag(v1)"))))
 Lcb = mpmath.mpc(mpmath.mpf(str(pari("real(v2)"))), mpmath.mpf(str(pari("imag(v2)"))))
 Ldh = A_const*Lc + B_const*Lcb
 return mpmath.re(mpmath.exp(I*theta_dh(t)) * Ldh)

def Z_hi(t):
 t = mpmath.mpf(t)
 s_str = f"(0.5) + ({mpmath.nstr(t, 70, strip_zeros=False)})*I"
 pari(f"v1 = lfun(Lchi_hi, {s_str})")
 pari(f"v2 = lfun(Lchibar_hi, {s_str})")
 Lc = mpmath.mpc(mpmath.mpf(str(pari("real(v1)"))), mpmath.mpf(str(pari("imag(v1)"))))
 Lcb = mpmath.mpc(mpmath.mpf(str(pari("real(v2)"))), mpmath.mpf(str(pari("imag(v2)"))))
 Ldh = A_const*Lc + B_const*Lcb
 return mpmath.re(mpmath.exp(I*theta_dh(t)) * Ldh)

for T in [100, 1000, 2000, 3000, 4000, 4500]:
 t0=time.time()
 v = Z_lo(T+0.123)
 dt = time.time()-t0
 t1=time.time()
 v2 = Z_hi(T+0.123)
 dt2 = time.time()-t1
 print(f"T={T}: Z_lo={dt*1000:.1f}ms, Z_hi={dt2*1000:.1f}ms, vals: {mpmath.nstr(v,6)} vs {mpmath.nstr(v2,12)}")

T=100: Z_lo=22.6ms, Z_hi=5.5ms, vals: 0.977986 vs 0.977985792611


 *** lfun: Warning: lfuninit: insufficient initialization.
 *** lfun: Warning: lfuninit: insufficient initialization.
 *** lfun: Warning: lfuninit: insufficient initialization.


T=1000: Z_lo=546.6ms, Z_hi=15.4ms, vals: 1.18746 vs 1.18745808904


 *** lfun: Warning: lfuninit: insufficient initialization.
 *** lfun: Warning: lfuninit: insufficient initialization.


 *** lfun: Warning: lfuninit: insufficient initialization.


T=2000: Z_lo=3512.9ms, Z_hi=49.9ms, vals: -0.809525 vs -0.809524680419


T=3000: Z_lo=147.1ms, Z_hi=140.5ms, vals: -1.06248 vs -1.06248221232
T=4000: Z_lo=136.1ms, Z_hi=135.3ms, vals: 2.26647 vs 2.26647464379


T=4500: Z_lo=137.0ms, Z_hi=139.8ms, vals: 0.935774 vs 0.935773786055


In [16]:
# The dps=30 lfuninit is INSUFFICIENT for high T. PARI auto-falls-back to recomputing -> very slow at large T.
# So scanning with dps=30 doesn't help.
# Conclusion: at high T we're stuck with ~140ms per evaluation at sufficient precision.
# 
# Better strategy: use Gram point or known interlacing? L_DH has off-line zeros so Gram points don't work cleanly.
# Better: scan with adaptive step. At spacing of ~0.7 use h=0.3-0.5. 
# Total range T=0..~4500, with h=0.5 = 9000 evals. Plus refinements ~5000*15 = 75000 evals.
# At 140ms avg: 84000 * 0.14 = 11760s = 3.27 hours. Tight but possible.
# 
# Actually most evals are at lower T. Mean cost might be ~80ms. 84000 * 0.08 = 112 minutes. OK.

# But h=0.5 might miss closely spaced zeros. Let me check: known L_DH zero spacing - they sometimes come in close pairs (due to off-line behavior). 
# Reduce to h=0.2 to be safe: ~22500 scans, plus refinement. Still feasible.

# Actually safer: h=0.1 initially. Total = 45000 + 75000 = 120,000 evals * 0.1s avg = 200 minutes = 3.3 hours.
# We need to be strategic. Let me first check timing at dps=80 with set_height matching actual T.

# Re-init with smaller T_max range: PARI's lfuninit at T=5200 may have built unnecessarily many coefficients. Let's see.
# The cost should scale with sqrt(N) for evaluation. The dominant cost depends on conductor and T.

# Let me try a smaller chunk: build lfuninit at T_max=1000 first, scan 0..900, then increase.
# This won't help in long run but will be MUCH faster in early scans.

# Actually best path: use the existing _hi evaluator. Reality check is fast at lower T because lfuninit precomputes asymptotic. Time grows with T.

# Let's estimate total scan time accurately. Sample times: at T=4000-4500: 140ms. At T=2000: 50ms. At T=1000: 15ms. T=100: 5ms.
# Integral: roughly avg over 0..4500 considering distribution.

import numpy as np
# Sample many points
ts = np.linspace(50, 4500, 20)
times = []
for tt in ts:
 t0=time.time()
 v = Z_hi(tt)
 times.append(time.time()-t0)
print(list(zip(ts.tolist(), times)))
print("avg ms:", np.mean(times)*1000)
print("Estimated scan with h=0.1, N=45000: ", 45000*np.mean(times)/60, "minutes")

[(50.0, 0.005348682403564453), (284.2105263157895, 0.007590532302856445), (518.421052631579, 0.011422872543334961), (752.6315789473684, 0.011672019958496094), (986.8421052631579, 0.014084100723266602), (1221.0526315789475, 0.01708674430847168), (1455.2631578947369, 0.020345687866210938), (1689.4736842105262, 0.03713250160217285), (1923.6842105263158, 0.02892160415649414), (2157.8947368421054, 0.05390310287475586), (2392.105263157895, 0.03993511199951172), (2626.315789473684, 0.13579320907592773), (2860.5263157894738, 0.13603782653808594), (3094.7368421052633, 0.13646268844604492), (3328.9473684210525, 0.13551092147827148), (3563.157894736842, 0.14145612716674805), (3797.3684210526317, 0.14439749717712402), (4031.5789473684213, 0.136033296585083), (4265.789473684211, 0.1383819580078125), (4500.0, 0.13991189002990723)]
avg ms: 74.57141876220703
Estimated scan with h=0.1, N=45000: 55.92856407165527 minutes


In [17]:
# OK so avg ~75ms. Scan 45000 evals = ~56 minutes. Refinement: 5000 zeros * ~15 evals = 75000 * 75ms = 94 minutes.
# Total ~ 150 minutes = 2.5 hours. Feasible.
# 
# Let me check why time SUDDENLY jumps at T=~2500 from ~50ms to ~135ms. Probably lfuninit "phases" - precomputed for [5200].
# It might help to lfuninit at smaller T_max for the lower-range portion. But complexity not worth it. Let's proceed.

# But to be safer, let me first try with a smaller initial lfuninit, see if early range is faster:
# Actually we built lfuninit_hi at T=5200 already, and got 5ms at T=50. So nothing to change.

# Strategy details:
# 1. Scan starting from t=10 (avoid singularity behaviors near 0).
# 2. Step h=0.15 initially. Adaptive: if |Z| is small (<1), use smaller step.
# 3. Detect sign change -> bracket [a,b].
# 4. Refine: use mpmath.findroot with secant or anderson method, brackets [a,b], to dps=50 precision.
# 5. Checkpoint every 250.

# First, let me find the first zero of L_DH to confirm we start right.
# Known: first Davenport-Heilbronn zero is around t ~ 9.7-10 or so... let's scan from t=0.5:
for t in np.arange(0.5, 25, 0.5):
 v = Z_hi(t)
 print(f"t={t:.1f}: Z={float(v):+.4f}")

t=0.5: Z=+0.8693
t=1.0: Z=+0.9866
t=1.5: Z=+1.1442
t=2.0: Z=+1.3035
t=2.5: Z=+1.4231
t=3.0: Z=+1.4594
t=3.5: Z=+1.3697
t=4.0: Z=+1.1208
t=4.5: Z=+0.6991
t=5.0: Z=+0.1222
t=5.5: Z=-0.5544
t=6.0: Z=-1.2364
t=6.5: Z=-1.8013
t=7.0: Z=-2.1201
t=7.5: Z=-2.0888
t=8.0: Z=-1.6617
t=8.5: Z=-0.8786
t=9.0: Z=+0.1259
t=9.5: Z=+1.1400
t=10.0: Z=+1.9194
t=10.5: Z=+2.2539
t=11.0: Z=+2.0362
t=11.5: Z=+1.3116
t=12.0: Z=+0.2856
t=12.5: Z=-0.7225
t=13.0: Z=-1.3736
t=13.5: Z=-1.4255
t=14.0: Z=-0.8312
t=14.5: Z=+0.2221
t=15.0: Z=+1.3584
t=15.5: Z=+2.1402
t=16.0: Z=+2.2355
t=16.5: Z=+1.5630
t=17.0: Z=+0.3499
t=17.5: Z=-0.9344
t=18.0: Z=-1.7562
t=18.5: Z=-1.7418
t=19.0: Z=-0.8525
t=19.5: Z=+0.5753
t=20.0: Z=+1.9580


t=20.5: Z=+2.7180
t=21.0: Z=+2.5528
t=21.5: Z=+1.5908
t=22.0: Z=+0.3470
t=22.5: Z=-0.5183
t=23.0: Z=-0.5296
t=23.5: Z=+0.3472
t=24.0: Z=+1.6631
t=24.5: Z=+2.6986


In [18]:
# Sign changes detected: between 5.0-5.5 (pos->neg), 8.5-9.0 (neg->pos), 12.0-12.5, 14.0-14.5, 17.0-17.5, 19.0-19.5, 22.0-22.5, 23.0-23.5...
# Spacing about 1.5-3 at low T. Good.

# Note: there's an issue near 22.5-23.5 - close zeros (pair). Step 0.5 caught them but step 0.7 might miss.
# For L_DH there are known close pairs. Use h=0.15 to be very safe.

# Let me check refinement speed.
import mpmath
def find_zero_in_bracket(a, b, fa, fb):
 """Refine zero in (a,b) where Z(a)=fa, Z(b)=fb of opposite signs."""
 # mpmath.findroot with bisection-like method
 return mpmath.findroot(Z_hi, (mpmath.mpf(a), mpmath.mpf(b)), solver='anderson')

t0=time.time()
z = find_zero_in_bracket(5.0, 5.5, Z_hi(5.0), Z_hi(5.5))
print(f"First zero: {z}, took {time.time()-t0:.3f}s")
print("Z at zero:", Z_hi(z))

First zero: 5.0941598445710949256987955170797974750670744531091, took 0.063s
Z at zero: -3.3373695530762336043119053360552788271641047694223e-51


In [19]:
# Excellent! First zero at t=5.094, refined to 50dps in 63ms.
# Let me check known: first zero of Davenport-Heilbronn is t ≈ 5.0942... ? Let me verify against literature.
# Spira 1968 mentions the zeros. Anyway, our function works.

# Set up the production script.
import os, time, json
import numpy as np
import mpmath

WORK_DIR = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d'
CHECKPOINT_DIR = os.path.join(WORK_DIR, 'checkpoints_ldh')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

mpmath.mp.dps = 50

# We need a global state to track progress
state = {
 'zeros': [], # list of mpmath.mpf strings (decimal, 50dp)
 'last_t': None, # last t scanned
 'last_sign': None, # sign at last_t
 'n_scans': 0,
 'n_zeros': 0,
}

def save_checkpoint(state, label):
 path = os.path.join(CHECKPOINT_DIR, f'ldh_ckpt_{label}.npy')
 arr = np.array(state['zeros'], dtype=object)
 np.save(path, arr, allow_pickle=True)
 # Also save meta
 meta = {'n_zeros': state['n_zeros'], 'last_t': str(state['last_t']), 
 'last_sign': state['last_sign'], 'n_scans': state['n_scans']}
 with open(os.path.join(CHECKPOINT_DIR, f'ldh_ckpt_{label}_meta.json'), 'w') as f:
 json.dump(meta, f)
 return path

print("Setup complete")

Setup complete


In [20]:
# Pre-flight: test refinement quality and edge cases.
# Test: bracket with very close zeros nearby. Use Z_hi exclusively (no Z_lo benefit).

# Let me also verify mpmath.findroot with anderson handles brackets robustly.
# Alternative: use 'bisect' (with single bracket), which is guaranteed convergent.

# mpmath.findroot solvers that accept brackets: 'anderson', 'illinois', 'pegasus', 'ridder', 'bisect'
# 'illinois' is usually fast and robust for brackets.

# Use 'illinois':
t0=time.time()
z = mpmath.findroot(Z_hi, (mpmath.mpf("5.0"), mpmath.mpf("5.5")), solver='illinois')
print(f"illinois: {z}, took {time.time()-t0:.3f}s, Z={mpmath.nstr(Z_hi(z),5)}")

t0=time.time()
z = mpmath.findroot(Z_hi, (mpmath.mpf("5.0"), mpmath.mpf("5.5")), solver='pegasus')
print(f"pegasus: {z}, took {time.time()-t0:.3f}s, Z={mpmath.nstr(Z_hi(z),5)}")

t0=time.time()
z = mpmath.findroot(Z_hi, (mpmath.mpf("5.0"), mpmath.mpf("5.5")), solver='anderson')
print(f"anderson: {z}, took {time.time()-t0:.3f}s, Z={mpmath.nstr(Z_hi(z),5)}")

illinois: 5.0941598445710949256987955170797974750670744531091, took 0.066s, Z=-3.3374e-51
pegasus: 5.0941598445710949256987955170797974750670744531091, took 0.054s, Z=-3.3374e-51
anderson: 5.0941598445710949256987955170797974750670744531091, took 0.051s, Z=-3.3374e-51


In [21]:
# anderson is fast. Use it.
# Now: write the main loop. Strategy:
# - Scan t from 1.0 upward with step h. 
# - Keep track of sign(Z(t)) at previous t.
# - On sign change, refine bracket -> append zero.
# - Adaptive step: if |Z| is large, take big step; if small, shrink.
# - But to be safe just use fixed h=0.1.
# - Checkpoint every 250 zeros.

# Production loop with safety:
import numpy as np
import mpmath, time, os, json

def sign(x):
 return 1 if x > 0 else (-1 if x < 0 else 0)

zeros = []
n_target = 5000
h = mpmath.mpf("0.1") # step
t_start = mpmath.mpf("1.0")

t_prev = t_start
z_prev = Z_hi(t_prev)
s_prev = sign(z_prev)

start_time = time.time()
last_report = start_time
n_evals = 1

def refine(a, b):
 """Refine root in bracket [a, b] using anderson."""
 return mpmath.findroot(Z_hi, (a, b), solver='anderson')

# Just test a small run to make sure logic works
test_max_zeros = 20
t = t_prev
while len(zeros) < test_max_zeros:
 t_next = t + h
 z_next = Z_hi(t_next)
 n_evals += 1
 s_next = sign(z_next)
 if s_next != s_prev and s_prev != 0:
 # Bracket [t, t_next]
 root = refine(t, t_next)
 zeros.append(root)
 t = t_next
 s_prev = s_next
 z_prev = z_next

print(f"Found {len(zeros)} zeros, n_evals={n_evals}, time={time.time()-start_time:.1f}s")
for i,z in enumerate(zeros[:10]):
 print(i+1, mpmath.nstr(z, 15))

Found 20 zeros, n_evals=456, time=3.1s
1 5.09415984457109
2 8.93991440809652
3 12.1335454258007
4 14.4040031122775
5 17.1302394005725
6 19.3088001742281
7 22.159707765094
8 23.3453701120268
9 26.0949673462454
10 27.9237988215997


In [22]:
# Good! Note zeros 7,8 are quite close (22.16, 23.35) - spacing 1.19. Step h=0.1 catches them.
# 
# But wait: are zeros 7, 8 actually close enough that h=0.1 could miss them in pair? Spacing is 1.19 here, safe.
# However at higher T closely spaced pairs known. Step h=0.1 should be safe based on typical L_DH zero spacing.

# Issue: 456 evals for 20 zeros, ratio ~23 evals per zero (scan + refine). 
# Refine took ~7-8 evals each, scan = ~16 evals.

# For 5000 zeros at h=0.1: total scan range ~4500. Scan evals = 4500/0.1 = 45000. Plus refinement ~50000 evals. Total ~100000.
# At avg 75ms = 125 minutes ~ 2 hours. Good.

# Let me also handle the edge case where Z=0 exactly (extremely unlikely with floats but possible).
# And handle: if step skips multiple zeros (multiple sign changes between t and t+h), we'd miss.
# Safety: detect anomaly by checking that we don't have refined root very near to previous, and that count grows monotonically.

# Now run the full computation. To enable progress reporting and checkpointing, do it in one cell with periodic output.
# But we want to checkpoint to disk in case of failure.

# Let me move to a more efficient main loop:
import numpy as np, mpmath, time, os, json

mpmath.mp.dps = 50
WORK_DIR = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d'
CHECKPOINT_DIR = os.path.join(WORK_DIR, 'checkpoints_ldh')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_ckpt(zeros, label, t_last, n_evals):
 arr = np.array([mpmath.nstr(z, 50, strip_zeros=False) for z in zeros], dtype=object)
 np.save(os.path.join(CHECKPOINT_DIR, f'ldh_ckpt_{label}.npy'), arr, allow_pickle=True)
 meta = {'n_zeros': len(zeros), 't_last': mpmath.nstr(t_last, 30), 'n_evals': n_evals}
 with open(os.path.join(CHECKPOINT_DIR, f'ldh_ckpt_{label}_meta.json'), 'w') as f:
 json.dump(meta, f)

print("Ready to run main loop")

Ready to run main loop


In [23]:
# Final preparation. Let's now run the full loop. Estimated time ~2 hours.
# We need to be careful about runtime budget (3600s total). 2 hours = 7200s > 3600s!
# 
# Wait: I have 3600s = 60 minutes. We've already used ~5min. So we have ~55 min budget for computation in this notebook.
# That's only enough to get partially done at this pace.
# 
# Let me reduce step h and use adaptive scan to be efficient.
# Actually the bigger speed gain: skip refinement (which uses ~7 expensive evals at high T = 1s per zero ~ 80 min for refinement alone).
# 
# Better idea: delegate the heavy computation to an analysis sub-agent which has more resources/time budget.

import os
print("Total time used so far in this notebook can be tracked but limit is 3600s.")
# Let's delegate to a sub-agent.

Total time used so far in this notebook can be tracked but limit is 3600s.


In [24]:
# Before delegating, let me see if I can complete it locally. Total budget = 3600s.
# Let me actually try to be very fast: process up to N zeros within available time.
# 
# Reality check: total evals needed = ~120,000 at avg ~75ms = 9000s = 2.5 hours.
# 
# This is too long for one notebook session. Let me delegate to an analysis sub-agent.
# 
# I'll write a Python script that does the full computation, then delegate execution.

script = r'''#!/usr/bin/env python
"""
Generate the first 5000 critical-line zeros of the Davenport-Heilbronn L-function.
Uses a fast PARI/GP-based Z_DH evaluator at dps=80 internal precision; refines roots
to dps=50 with mpmath.findroot (anderson). Checkpoints every 250 zeros.

Outputs:
 - ldh_zeros_5000_dps50.npy final array of strings (50 dp)
 - checkpoints_ldh/ldh_ckpt_*.npy intermediate checkpoints
 - run_log.txt progress log
"""
import os, sys, time, json
import numpy as np
import mpmath
import cypari2

WORK_DIR = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d'
os.chdir(WORK_DIR)
CKPT_DIR = os.path.join(WORK_DIR, 'checkpoints_ldh')
os.makedirs(CKPT_DIR, exist_ok=True)
LOG_PATH = os.path.join(WORK_DIR, 'run_log.txt')

def log(msg):
 line = f"[{time.strftime('%H:%M:%S')}] {msg}"
 print(line, flush=True)
 with open(LOG_PATH, 'a') as f:
 f.write(line + '\n')

log("Starting L_DH zero generation")

# === Initialize PARI ===
mpmath.mp.dps = 50
pari = cypari2.Pari()
pari.allocatemem(1024*1024*1024) # 1 GB stack

# Target imaginary range: expected 5000th zero near T~4400. Use T_max=5200 for safety.
T_MAX = 5200
DPS_PARI = 80 # internal PARI precision

pari(f"default(realprecision, {DPS_PARI})")
pari("gz5 = znstar(5, 1)")
pari("Lchi = lfuncreate([gz5, [1]])")
pari("Lchibar = lfuncreate([gz5, [3]])")
log(f"Building lfuninit at dps={DPS_PARI}, T_max={T_MAX} ...")
t0 = time.time()
pari(f"Lchi_hi = lfuninit(Lchi, [{T_MAX}])")
pari(f"Lchibar_hi = lfuninit(Lchibar, [{T_MAX}])")
log(f"lfuninit done in {time.time()-t0:.1f}s")

# Davenport-Heilbronn constants in mpmath
sqrt5 = mpmath.sqrt(5)
xi_dh = (mpmath.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
I = mpmath.mpc(0, 1)
A_const = (1 - I*xi_dh)/2
B_const = (1 + I*xi_dh)/2

def theta_dh(t):
 t = mpmath.mpf(t)
 return (t/2)*mpmath.log(5/mpmath.pi) + mpmath.im(mpmath.loggamma(mpmath.mpc(mpmath.mpf("0.75"), t/2)))

def Z(t):
 """Z_DH(t) via PARI fast evaluator (dps=80 internal, returns mpf at 50dp)."""
 t = mpmath.mpf(t)
 s_str = f"(0.5) + ({mpmath.nstr(t, 70, strip_zeros=False)})*I"
 pari(f"v1 = lfun(Lchi_hi, {s_str})")
 pari(f"v2 = lfun(Lchibar_hi, {s_str})")
 Lc = mpmath.mpc(mpmath.mpf(str(pari("real(v1)"))), mpmath.mpf(str(pari("imag(v1)"))))
 Lcb = mpmath.mpc(mpmath.mpf(str(pari("real(v2)"))), mpmath.mpf(str(pari("imag(v2)"))))
 Ldh = A_const*Lc + B_const*Lcb
 return mpmath.re(mpmath.exp(I*theta_dh(t)) * Ldh)

def sgn(x):
 return 1 if x > 0 else (-1 if x < 0 else 0)

# === Resume from checkpoint if available ===
zeros = []
t_start = mpmath.mpf("1.0")

ckpts = sorted([f for f in os.listdir(CKPT_DIR) if f.startswith('ldh_ckpt_') and f.endswith('.npy')])
if ckpts:
 last = ckpts[-1]
 label = last.replace('ldh_ckpt_','').replace('.npy','')
 meta_path = os.path.join(CKPT_DIR, f'ldh_ckpt_{label}_meta.json')
 if os.path.exists(meta_path):
 arr = np.load(os.path.join(CKPT_DIR, last), allow_pickle=True)
 zeros = [mpmath.mpf(str(s)) for s in arr]
 with open(meta_path) as f:
 meta = json.load(f)
 t_start = mpmath.mpf(meta['t_last'])
 log(f"Resumed from {last}: {len(zeros)} zeros, t_last={float(t_start):.4f}")

# === Main scan loop ===
h = mpmath.mpf("0.1")
N_TARGET = 5000
t = t_start
z_prev = Z(t)
s_prev = sgn(z_prev)
n_evals = 1
last_ckpt = (len(zeros) // 250) * 250

start = time.time()
last_report_time = start

while len(zeros) < N_TARGET:
 t_next = t + h
 z_next = Z(t_next)
 n_evals += 1
 s_next = sgn(z_next)
 if s_next != s_prev and s_prev != 0:
 # Bracket found. Refine.
 try:
 root = mpmath.findroot(Z, (t, t_next), solver='anderson')
 except Exception as e:
 log(f"Anderson failed at [{float(t)},{float(t_next)}]: {e}; trying illinois")
 try:
 root = mpmath.findroot(Z, (t, t_next), solver='illinois')
 except Exception as e2:
 log(f"Illinois also failed: {e2}; trying bisect")
 root = mpmath.findroot(Z, (t, t_next), solver='bisect')
 n_evals += 12 # approximate
 # Sanity check
 if zeros and root <= zeros[-1]:
 log(f"WARNING: non-monotonic root {float(root)} <= prev {float(zeros[-1])}; skipping")
 else:
 zeros.append(root)
 if len(zeros) // 250 > last_ckpt // 250:
 last_ckpt = (len(zeros) // 250) * 250
 label = f"{last_ckpt:05d}"
 arr = np.array([mpmath.nstr(z, 50, strip_zeros=False) for z in zeros], dtype=object)
 np.save(os.path.join(CKPT_DIR, f'ldh_ckpt_{label}.npy'), arr, allow_pickle=True)
 meta = {'n_zeros': len(zeros), 't_last': mpmath.nstr(t_next, 30), 'n_evals': n_evals}
 with open(os.path.join(CKPT_DIR, f'ldh_ckpt_{label}_meta.json'), 'w') as f:
 json.dump(meta, f)
 elapsed = time.time() - start
 rate = len(zeros) / elapsed if elapsed > 0 else 0
 log(f"Checkpoint {label}: {len(zeros)} zeros, t={float(t_next):.2f}, "
 f"elapsed={elapsed/60:.1f}min, rate={rate:.2f} z/s, n_evals={n_evals}")
 t = t_next
 s_prev = s_next
 z_prev = z_next
 # Periodic progress (not for checkpoint)
 if time.time() - last_report_time > 60:
 last_report_time = time.time()
 elapsed = time.time() - start
 log(f"PROGRESS: {len(zeros)} zeros, t={float(t):.2f}, elapsed={elapsed/60:.1f}min")

log(f"DONE. Found {len(zeros)} zeros in {(time.time()-start)/60:.1f} minutes")

# === Verify and save final ===
# Monotonic check
sorted_ok = all(zeros[i] < zeros[i+1] for i in range(len(zeros)-1))
log(f"Strictly monotonic: {sorted_ok}")
log(f"Total zeros: {len(zeros)}")
log(f"First zero: {mpmath.nstr(zeros[0], 50)}")
log(f"Last zero: {mpmath.nstr(zeros[-1], 50)}")

# Save
arr = np.array([mpmath.nstr(z, 50, strip_zeros=False) for z in zeros], dtype=object)
np.save(os.path.join(WORK_DIR, 'ldh_zeros_5000_dps50.npy'), arr, allow_pickle=True)
log(f"Saved to ldh_zeros_5000_dps50.npy")
'''

script_path = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/generate_ldh_zeros.py'
with open(script_path, 'w') as f:
 f.write(script)
print(f"Wrote {script_path}")
print(f"Size: {len(script)} chars")

Wrote /workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/generate_ldh_zeros.py
Size: 6099 chars


In [25]:
# Now delegate to an analysis sub-agent to run this script.
from e14c.subagents import submit_task, check_task

query = """Execute the L_DH zero generation script at `/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/generate_ldh_zeros.py`.

Context:
- This computes the first 5000 critical-line zeros of the Davenport-Heilbronn L-function L_DH
- Uses a fast PARI/GP evaluator (via cypari2) wrapping `lfuninit` for the mod-5 Dirichlet characters
- Refines roots to 50 decimal places (dps=50) using mpmath.findroot with anderson method
- Internal PARI precision is dps=80 to ensure 50-digit accuracy of refined roots
- Estimated runtime: 2-3 hours
- Checkpoints every 250 zeros to /workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/checkpoints_ldh/
- Final output: /workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/ldh_zeros_5000_dps50.npy
- Log file: /workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/run_log.txt

Steps:
1. Install cypari2 if not present (`pip install cypari2`).
2. Run the script: `python /workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/generate_ldh_zeros.py`
3. The script supports resuming from checkpoint if interrupted.
4. After completion, verify:
 - `ldh_zeros_5000_dps50.npy` exists
 - Contains exactly 5000 entries
 - Values are strictly monotonically increasing
 - All values stored as strings with 50 decimal places
5. Report final results: count, first zero, last zero, total runtime, and confirmation of strict monotonicity.

If the script encounters issues, debug and report. Do not modify the core mathematical algorithm without justification.
"""

task_id = await submit_task(query, "rh-zeros-ldh", agent_type="analysis")
print("Submitted task:", task_id)

ValueError: Unknown agent_type 'analysis'. Valid values: ['data-retrieval']

In [26]:
# Only data-retrieval available. I need to run this myself within the budget.
# Strategy: optimize and run partially, save what we can.
# 
# Total budget: 3600s. Used so far ~7min. Remaining ~52 min for computation.
# At ~75ms per eval avg, with 23 evals per zero, that's ~1.7s per zero.
# In 52 min = 3120s we can do ~1800 zeros. Not 5000.
# 
# Need to be more efficient. Key insight: at low T, evaluations are very fast (5-15ms).
# At high T (>2500), each eval is 140ms.
# 
# OPTIMIZATION: use Z_lo with lfuninit at smaller T to be fast in lower range.
# But that's complicated. Let's check: where are we limited?
# 
# Actually the 3600s constraint is total runtime. But the cell can keep running. Let me just start running.
# I'll set a hard time limit of ~45 minutes and save whatever we get.
# 
# However, the hypothesis explicitly says "under 4 hours" feasibility. So we need to actually demonstrate completion.
# Without a sub-agent we can't do this in one notebook session given the 60-min limit.
# 
# Let me check: can I do the computation in background/disk script while continuing?
import subprocess
# Run in background
proc = subprocess.Popen(
 ['python', '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/generate_ldh_zeros.py'],
 stdout=open('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/script_stdout.log','w'),
 stderr=subprocess.STDOUT,
)
print("Started background script, PID:", proc.pid)

Started background script, PID: 67


In [27]:
import time, os
time.sleep(60)
# Check progress
print("Process still running:", proc.poll() is None)
log_path = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/run_log.txt'
if os.path.exists(log_path):
 with open(log_path) as f:
 print(f.read())
else:
 print("No log yet")
print("---")
stdout_path = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/script_stdout.log'
if os.path.exists(stdout_path):
 with open(stdout_path) as f:
 s = f.read()
 print(s[-2000:])

Process still running: False
[06:42:15] Starting L_DH zero generation
[06:42:15] Building lfuninit at dps=80, T_max=5200 ...
[06:42:22] lfuninit done in 6.9s
[06:42:25] Anderson failed at [48.4,48.5]: could not convert string to float: '1.0729487556423575868831673167475960789370875609445452356669409655434719751938206 e-5'; trying illinois
[06:42:25] Illinois also failed: could not convert string to float: '1.0729487556423575868831673167475960789370875609445452356669409655434719751938206 e-5'; trying bisect

---
app/kernel_env/lib/python3.13/site-packages/mpmath/libmp/libmpf.py", line 1294, in str_to_man_exp
 float(x)
 ~~~~~^^^
ValueError: could not convert string to float: '1.0729487556423575868831673167475960789370875609445452356669409655434719751938206 e-5'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
 File "/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/generate_ldh_zeros.py", line 119, in <module>
 root = mpmath.findroot(

In [28]:
# Bug: PARI returns "1.0729... e-5" with a SPACE before "e". mpmath can't parse.
# Need to strip the space.

# Fix the str conversion:
def pari_to_mpf(name_or_val):
 s = str(name_or_val).strip().replace(' ', '')
 return mpmath.mpf(s)

# Test
import cypari2
pari2 = cypari2.Pari()
pari2(f"default(realprecision, 80)")
v = pari2("0.00001072948755642357586")
print(repr(str(v)))

'1.0729487556423575860000000000000000000000000000000000000000000000000000000000000 E-5'


In [29]:
# Yes, PARI inserts space before E. Let me fix the script and restart.
# Also handle uppercase E:
import re
def fix_pari_str(s):
 s = str(s).strip()
 # Remove any whitespace and lowercase
 s = re.sub(r'\s+', '', s)
 s = s.replace('E', 'e')
 return s

s = "1.0729487556423575860000000000000000000000000000000000000000000000000000000000000 E-5"
print(repr(fix_pari_str(s)))
print(mpmath.mpf(fix_pari_str(s)))

'1.0729487556423575860000000000000000000000000000000000000000000000000000000000000e-5'
0.00001072948755642357586


In [30]:
# Good. Update the script and restart.
new_script = r'''#!/usr/bin/env python
"""
Generate the first 5000 critical-line zeros of the Davenport-Heilbronn L-function.
"""
import os, sys, time, json, re
import numpy as np
import mpmath
import cypari2

WORK_DIR = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d'
os.chdir(WORK_DIR)
CKPT_DIR = os.path.join(WORK_DIR, 'checkpoints_ldh')
os.makedirs(CKPT_DIR, exist_ok=True)
LOG_PATH = os.path.join(WORK_DIR, 'run_log.txt')

def log(msg):
 line = f"[{time.strftime('%H:%M:%S')}] {msg}"
 print(line, flush=True)
 with open(LOG_PATH, 'a') as f:
 f.write(line + '\n')

log("Starting L_DH zero generation")

mpmath.mp.dps = 50
pari = cypari2.Pari()
pari.allocatemem(1024*1024*1024)

T_MAX = 5200
DPS_PARI = 80

pari(f"default(realprecision, {DPS_PARI})")
pari("gz5 = znstar(5, 1)")
pari("Lchi = lfuncreate([gz5, [1]])")
pari("Lchibar = lfuncreate([gz5, [3]])")
log(f"Building lfuninit at dps={DPS_PARI}, T_max={T_MAX} ...")
t0 = time.time()
pari(f"Lchi_hi = lfuninit(Lchi, [{T_MAX}])")
pari(f"Lchibar_hi = lfuninit(Lchibar, [{T_MAX}])")
log(f"lfuninit done in {time.time()-t0:.1f}s")

sqrt5 = mpmath.sqrt(5)
xi_dh = (mpmath.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
I = mpmath.mpc(0, 1)
A_const = (1 - I*xi_dh)/2
B_const = (1 + I*xi_dh)/2

def theta_dh(t):
 t = mpmath.mpf(t)
 return (t/2)*mpmath.log(5/mpmath.pi) + mpmath.im(mpmath.loggamma(mpmath.mpc(mpmath.mpf("0.75"), t/2)))

def fix_pari_str(s):
 s = re.sub(r'\s+', '', str(s).strip())
 s = s.replace('E', 'e')
 return s

def Z(t):
 t = mpmath.mpf(t)
 s_str = f"(0.5) + ({mpmath.nstr(t, 70, strip_zeros=False)})*I"
 pari(f"v1 = lfun(Lchi_hi, {s_str})")
 pari(f"v2 = lfun(Lchibar_hi, {s_str})")
 Lc = mpmath.mpc(mpmath.mpf(fix_pari_str(pari("real(v1)"))),
 mpmath.mpf(fix_pari_str(pari("imag(v1)"))))
 Lcb = mpmath.mpc(mpmath.mpf(fix_pari_str(pari("real(v2)"))),
 mpmath.mpf(fix_pari_str(pari("imag(v2)"))))
 Ldh = A_const*Lc + B_const*Lcb
 return mpmath.re(mpmath.exp(I*theta_dh(t)) * Ldh)

def sgn(x):
 return 1 if x > 0 else (-1 if x < 0 else 0)

# Resume
zeros = []
t_start = mpmath.mpf("1.0")
ckpts = sorted([f for f in os.listdir(CKPT_DIR) if f.startswith('ldh_ckpt_') and f.endswith('.npy')])
if ckpts:
 last = ckpts[-1]
 label = last.replace('ldh_ckpt_','').replace('.npy','')
 meta_path = os.path.join(CKPT_DIR, f'ldh_ckpt_{label}_meta.json')
 if os.path.exists(meta_path):
 arr = np.load(os.path.join(CKPT_DIR, last), allow_pickle=True)
 zeros = [mpmath.mpf(str(s)) for s in arr]
 with open(meta_path) as f:
 meta = json.load(f)
 t_start = mpmath.mpf(meta['t_last'])
 log(f"Resumed from {last}: {len(zeros)} zeros, t_last={float(t_start):.4f}")

h = mpmath.mpf("0.1")
N_TARGET = 5000
t = t_start
z_prev = Z(t)
s_prev = sgn(z_prev)
n_evals = 1
last_ckpt = (len(zeros) // 250) * 250

start = time.time()
last_report_time = start

while len(zeros) < N_TARGET:
 t_next = t + h
 z_next = Z(t_next)
 n_evals += 1
 s_next = sgn(z_next)
 if s_next != s_prev and s_prev != 0:
 try:
 root = mpmath.findroot(Z, (t, t_next), solver='anderson')
 except Exception as e:
 log(f"Anderson failed at [{float(t)},{float(t_next)}]: {e}; trying illinois")
 try:
 root = mpmath.findroot(Z, (t, t_next), solver='illinois')
 except Exception as e2:
 log(f"Illinois failed: {e2}; trying bisect")
 root = mpmath.findroot(Z, (t, t_next), solver='bisect')
 n_evals += 12
 if zeros and root <= zeros[-1]:
 log(f"WARNING: non-monotonic root {float(root)} <= prev {float(zeros[-1])}; skipping")
 else:
 zeros.append(root)
 if len(zeros) // 250 > last_ckpt // 250:
 last_ckpt = (len(zeros) // 250) * 250
 label = f"{last_ckpt:05d}"
 arr = np.array([mpmath.nstr(z, 50, strip_zeros=False) for z in zeros], dtype=object)
 np.save(os.path.join(CKPT_DIR, f'ldh_ckpt_{label}.npy'), arr, allow_pickle=True)
 meta = {'n_zeros': len(zeros), 't_last': mpmath.nstr(t_next, 30), 'n_evals': n_evals}
 with open(os.path.join(CKPT_DIR, f'ldh_ckpt_{label}_meta.json'), 'w') as f:
 json.dump(meta, f)
 elapsed = time.time() - start
 rate = len(zeros) / elapsed if elapsed > 0 else 0
 log(f"Checkpoint {label}: {len(zeros)} zeros, t={float(t_next):.2f}, "
 f"elapsed={elapsed/60:.1f}min, rate={rate:.2f} z/s, n_evals={n_evals}")
 t = t_next
 s_prev = s_next
 z_prev = z_next
 if time.time() - last_report_time > 60:
 last_report_time = time.time()
 elapsed = time.time() - start
 log(f"PROGRESS: {len(zeros)} zeros, t={float(t):.2f}, elapsed={elapsed/60:.1f}min")

log(f"DONE. Found {len(zeros)} zeros in {(time.time()-start)/60:.1f} minutes")

sorted_ok = all(zeros[i] < zeros[i+1] for i in range(len(zeros)-1))
log(f"Strictly monotonic: {sorted_ok}")
log(f"Total zeros: {len(zeros)}")
log(f"First zero: {mpmath.nstr(zeros[0], 50)}")
log(f"Last zero: {mpmath.nstr(zeros[-1], 50)}")

arr = np.array([mpmath.nstr(z, 50, strip_zeros=False) for z in zeros], dtype=object)
np.save(os.path.join(WORK_DIR, 'ldh_zeros_5000_dps50.npy'), arr, allow_pickle=True)
log(f"Saved to ldh_zeros_5000_dps50.npy")
'''

with open('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/generate_ldh_zeros.py', 'w') as f:
 f.write(new_script)

# Clear previous checkpoints since we have only ~0
import shutil
ckpt_dir = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/checkpoints_ldh'
for f in os.listdir(ckpt_dir):
 os.remove(os.path.join(ckpt_dir, f))
print("Cleared checkpoints")

# Clear log
log_path = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/run_log.txt'
if os.path.exists(log_path):
 os.remove(log_path)

# Restart background process
proc = subprocess.Popen(
 ['python', '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/generate_ldh_zeros.py'],
 stdout=open('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/script_stdout.log','w'),
 stderr=subprocess.STDOUT,
)
print("Restarted, PID:", proc.pid)

Cleared checkpoints
Restarted, PID: 76


In [31]:
import time
time.sleep(120)
print("Process running:", proc.poll() is None)
with open('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/run_log.txt') as f:
 print(f.read())

Process running: True
[06:44:06] Starting L_DH zero generation
[06:44:07] Building lfuninit at dps=80, T_max=5200 ...
[06:44:13] lfuninit done in 6.9s
[06:44:49] Checkpoint 00250: 250 zeros, t=356.30, elapsed=0.6min, rate=7.02 z/s, n_evals=6554
[06:45:13] PROGRESS: 398 zeros, t=523.90, elapsed=1.0min
[06:45:34] Checkpoint 00500: 500 zeros, t=638.30, elapsed=1.3min, rate=6.18 z/s, n_evals=12374



In [32]:
# Excellent! Going much faster than I estimated. 500 zeros in 1.3 min = ~6 z/sec.
# At this rate, 5000 zeros in ~12.5 minutes! Let's keep monitoring.
time.sleep(120)
print("Process running:", proc.poll() is None)
with open('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/run_log.txt') as f:
 print(f.read())

Process running: True
[06:44:06] Starting L_DH zero generation
[06:44:07] Building lfuninit at dps=80, T_max=5200 ...
[06:44:13] lfuninit done in 6.9s
[06:44:49] Checkpoint 00250: 250 zeros, t=356.30, elapsed=0.6min, rate=7.02 z/s, n_evals=6554
[06:45:13] PROGRESS: 398 zeros, t=523.90, elapsed=1.0min
[06:45:34] Checkpoint 00500: 500 zeros, t=638.30, elapsed=1.3min, rate=6.18 z/s, n_evals=12374
[06:46:14] PROGRESS: 669 zeros, t=817.10, elapsed=2.0min
[06:46:35] Checkpoint 00750: 750 zeros, t=903.00, elapsed=2.4min, rate=5.29 z/s, n_evals=18021
[06:47:14] PROGRESS: 876 zeros, t=1031.50, elapsed=3.0min
[06:47:58] Checkpoint 01000: 1000 zeros, t=1152.10, elapsed=3.7min, rate=4.46 z/s, n_evals=23512
[06:48:14] PROGRESS: 1041 zeros, t=1193.40, elapsed=4.0min



In [33]:
# Going great. Rate slowing down at higher T as expected. Let's keep waiting.
time.sleep(240)
print("Process running:", proc.poll() is None)
with open('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/run_log.txt') as f:
 print(f.read())

Process running: True
[06:44:06] Starting L_DH zero generation
[06:44:07] Building lfuninit at dps=80, T_max=5200 ...
[06:44:13] lfuninit done in 6.9s
[06:44:49] Checkpoint 00250: 250 zeros, t=356.30, elapsed=0.6min, rate=7.02 z/s, n_evals=6554
[06:45:13] PROGRESS: 398 zeros, t=523.90, elapsed=1.0min
[06:45:34] Checkpoint 00500: 500 zeros, t=638.30, elapsed=1.3min, rate=6.18 z/s, n_evals=12374
[06:46:14] PROGRESS: 669 zeros, t=817.10, elapsed=2.0min
[06:46:35] Checkpoint 00750: 750 zeros, t=903.00, elapsed=2.4min, rate=5.29 z/s, n_evals=18021
[06:47:14] PROGRESS: 876 zeros, t=1031.50, elapsed=3.0min
[06:47:58] Checkpoint 01000: 1000 zeros, t=1152.10, elapsed=3.7min, rate=4.46 z/s, n_evals=23512
[06:48:14] PROGRESS: 1041 zeros, t=1193.40, elapsed=4.0min
[06:49:14] PROGRESS: 1178 zeros, t=1330.80, elapsed=5.0min
[06:49:49] Checkpoint 01250: 1250 zeros, t=1402.50, elapsed=5.6min, rate=3.72 z/s, n_evals=29016
[06:50:14] PROGRESS: 1295 zeros, t=1448.50, elapsed=6.0min
[06:51:14] PROGRESS: 1

In [34]:
# Rate dropping: 7 -> 6 -> 5 -> 4 -> 3 z/s as T grows. At rate ~2 z/s for last 3500 zeros that's 30+ min.
# Extrapolate: zero 5000 around t=4400. Current at t=1647 with 8 min. Time per increment of T: ~8/1647 -> ~0.005 min/T.
# But cost per T scales roughly linearly with T (since eval cost ~ sqrt(T) or so). 
# Let's keep waiting and re-estimate.
time.sleep(360)
print("Process running:", proc.poll() is None)
with open('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/run_log.txt') as f:
 print(f.read()[-3000:])

Process running: True
[06:44:06] Starting L_DH zero generation
[06:44:07] Building lfuninit at dps=80, T_max=5200 ...
[06:44:13] lfuninit done in 6.9s
[06:44:49] Checkpoint 00250: 250 zeros, t=356.30, elapsed=0.6min, rate=7.02 z/s, n_evals=6554
[06:45:13] PROGRESS: 398 zeros, t=523.90, elapsed=1.0min
[06:45:34] Checkpoint 00500: 500 zeros, t=638.30, elapsed=1.3min, rate=6.18 z/s, n_evals=12374
[06:46:14] PROGRESS: 669 zeros, t=817.10, elapsed=2.0min
[06:46:35] Checkpoint 00750: 750 zeros, t=903.00, elapsed=2.4min, rate=5.29 z/s, n_evals=18021
[06:47:14] PROGRESS: 876 zeros, t=1031.50, elapsed=3.0min
[06:47:58] Checkpoint 01000: 1000 zeros, t=1152.10, elapsed=3.7min, rate=4.46 z/s, n_evals=23512
[06:48:14] PROGRESS: 1041 zeros, t=1193.40, elapsed=4.0min
[06:49:14] PROGRESS: 1178 zeros, t=1330.80, elapsed=5.0min
[06:49:49] Checkpoint 01250: 1250 zeros, t=1402.50, elapsed=5.6min, rate=3.72 z/s, n_evals=29016
[06:50:14] PROGRESS: 1295 zeros, t=1448.50, elapsed=6.0min
[06:51:14] PROGRESS: 1

In [35]:
# Rate slowing as expected. 1877 zeros in 14 min. Remaining: 3123 zeros at slowing rate.
# Per minute we're getting ~60 zeros now. May drop to ~30-40 zeros/min at high T.
# Estimate: 3123 / 50 = ~62 more minutes. Total ~76 min. Over the 60-min runtime budget!
# 
# Need to keep monitoring. Worst case we have a checkpoint and report partial completion.
# 
# Let me wait further.
time.sleep(600)
with open('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/run_log.txt') as f:
 print(f.read()[-3000:])
print("Process running:", proc.poll() is None)

[06:44:06] Starting L_DH zero generation
[06:44:07] Building lfuninit at dps=80, T_max=5200 ...
[06:44:13] lfuninit done in 6.9s
[06:44:49] Checkpoint 00250: 250 zeros, t=356.30, elapsed=0.6min, rate=7.02 z/s, n_evals=6554
[06:45:13] PROGRESS: 398 zeros, t=523.90, elapsed=1.0min
[06:45:34] Checkpoint 00500: 500 zeros, t=638.30, elapsed=1.3min, rate=6.18 z/s, n_evals=12374
[06:46:14] PROGRESS: 669 zeros, t=817.10, elapsed=2.0min
[06:46:35] Checkpoint 00750: 750 zeros, t=903.00, elapsed=2.4min, rate=5.29 z/s, n_evals=18021
[06:47:14] PROGRESS: 876 zeros, t=1031.50, elapsed=3.0min
[06:47:58] Checkpoint 01000: 1000 zeros, t=1152.10, elapsed=3.7min, rate=4.46 z/s, n_evals=23512
[06:48:14] PROGRESS: 1041 zeros, t=1193.40, elapsed=4.0min
[06:49:14] PROGRESS: 1178 zeros, t=1330.80, elapsed=5.0min
[06:49:49] Checkpoint 01250: 1250 zeros, t=1402.50, elapsed=5.6min, rate=3.72 z/s, n_evals=29016
[06:50:14] PROGRESS: 1295 zeros, t=1448.50, elapsed=6.0min
[06:51:14] PROGRESS: 1400 zeros, t=1548.60, 

In [36]:
# 2374 zeros in 24 min. Rate dropping. Average for last 250: ~50/min = 0.83 z/s. Will continue to drop.
# Remaining: 2626 zeros. At current rate ~50/min that's ~52 min.
# Total: 24 + 52 = ~76 min. We're at ~25 min of notebook time used (5 min setup + 20 min waiting).
# 
# Notebook budget: 60 min = 3600s. We have ~35 min remaining of notebook time.
# Strategy: Wait more, then if not done, save checkpoint as the result and report status.
# 
# Actually wait - I think the runtime limit applies to MY session of cells. The background python process can keep running independently.
# But once I submit my answer, the agent session ends.
# 
# Let me keep waiting and hope it finishes.

time.sleep(600)
with open('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/run_log.txt') as f:
 print(f.read()[-2500:])
print("Process running:", proc.poll() is None)

os, t=1152.10, elapsed=3.7min, rate=4.46 z/s, n_evals=23512
[06:48:14] PROGRESS: 1041 zeros, t=1193.40, elapsed=4.0min
[06:49:14] PROGRESS: 1178 zeros, t=1330.80, elapsed=5.0min
[06:49:49] Checkpoint 01250: 1250 zeros, t=1402.50, elapsed=5.6min, rate=3.72 z/s, n_evals=29016
[06:50:14] PROGRESS: 1295 zeros, t=1448.50, elapsed=6.0min
[06:51:14] PROGRESS: 1400 zeros, t=1548.60, elapsed=7.0min
[06:52:14] PROGRESS: 1493 zeros, t=1640.90, elapsed=8.0min
[06:52:20] Checkpoint 01500: 1500 zeros, t=1646.80, elapsed=8.1min, rate=3.08 z/s, n_evals=34459
[06:53:14] PROGRESS: 1560 zeros, t=1712.10, elapsed=9.0min
[06:54:14] PROGRESS: 1628 zeros, t=1774.90, elapsed=10.0min
[06:55:14] PROGRESS: 1691 zeros, t=1836.90, elapsed=11.0min
[06:56:12] Checkpoint 01750: 1750 zeros, t=1893.10, elapsed=12.0min, rate=2.43 z/s, n_evals=39922
[06:56:15] PROGRESS: 1753 zeros, t=1895.50, elapsed=12.0min
[06:57:16] PROGRESS: 1820 zeros, t=1963.30, elapsed=13.0min
[06:58:16] PROGRESS: 1877 zeros, t=2016.90, elapsed=14

In [37]:
# Time is constrained. Rate has dropped to ~20 zeros/min. We have 2588 zeros at t=2667.
# We won't reach 5000 in the remaining ~17 min.
# 
# Let's wait a few more minutes for more progress and then submit with partial results.
# We have ~17 min remaining. Let's wait 10 minutes more then finalize.
import time
time.sleep(600)
with open('/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d/run_log.txt') as f:
 content = f.read()
print(content[-2000:])
print("Process running:", proc.poll() is None)

TimeoutError: Code execution timed out after 460.0 seconds